In [ ]:
import pandas as pd
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from config import URL, TOKEN

import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

itemCount = np.inf
data = {"entries": []}

offset = 0
steps = 1000

progress = None

while len(data["entries"]) < itemCount:

    response = requests.get(
        f"{URL}person/client/query?sort=personNr&limit={steps}&skip={offset}",
        headers=headers,
        verify=False
    )

    if response.status_code == 200:
        responseData = response.json()

        entries = responseData["entries"]

        # Total becomes known after first request
        if progress is None:
            itemCount = responseData["pagingInfo"]["itemCount"]

            progress = tqdm(
                total=itemCount,
                desc="Downloading clients",
                unit="entries"
            )

        data["entries"].extend(entries)

        # Advance by actual number received
        progress.update(len(entries))

        offset += steps

    else:
        print(response.json())
        break

if progress is not None:
    progress.close()

In [ ]:
exceptions = [
	"metadata",
	"locality",
	"postalcode",
	"contactoptions",
	"personFullName",
	"personFullNameNoTitle",
	"tenantid",
	"persontype",
	"adresse",
	"mailAddressing",
	"socialInsuranceNumber",
	"comments",
	"givenName",
	"familyName",
	"primaryEmailAddress",
	'addresses',
	'primaryPhoneNumber',
	'address',
	'residentialAddress',
	'address',
	'geoLocation',
	'billingAddress',
	'serviceAddress',
	'legalAddress',
	'businessAddress',
	'emailAddresses',
	'phoneNumbers',
	'personFullNameNoTitle',
	'personFullName',
	'mailAddressing',
	'postalAddress',
	'contractualAddressing',
	'caatsDateOfBirth',
	'gender',
	'locations',
	'bankDetails',
	'personStatus',
	"chapterId",
	"language",
	"ordinal",
	"sectionId",
	"academicTitlePrefix",
	"personNr",
	"content",
	"consecutiveNumber",
	'Klient:innen_Empfohlen_Ja',
	'Klient:innen_Erstrkontakt_erfassen_Ja',
	'Klient:innen_Bew_Ein_Ja',
	'abrech-akonto',
	'abrech-buerge-hinterlegt',
	'pers-visite-keine',
	'medizinische-delegation-hochgeladen',
	'medizinische-delegation-nicht-notwendig',
	'Klient:innen_Medizinische_Delegation_Ja',
	'pflegerische-delegation-hochgeladen',
	'pflegerische-delegation-nicht-notwendig',
	'Klient:innen_Pflegerische_Delegation_JA',
	"admin-beruf",
	'pflegevisite-letzte-date',
	'pflegevisite-letzte-dgkp',
	'pflegerisch-person',
	'erstkontakt-bemerkung',
	"admin-kooperationspartner",
	"orgRef",
	"admin.gsber",
	"admin-bundesland",
	"avatarFileId",
	"vertragsdatum",
	"betreuungsbeginn"
]

exceptions = [x.lower() for x in exceptions]

In [ ]:
def extractStatement(statement):
    field = {}
    if statement["statementId"].lower() in exceptions:
        return field

    match statement["answerScheme"]:
        case 1:
            field[statement["statementId"]] = statement["answerValue"]["displayText"]
        case 2:
            #do nothing
            field = {}
        case 3:
            field[statement["statementId"]] = statement["answerYesno"] == 2
        case 4:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False

        case 5:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False
        case 8:
            if "answerDateTime" in statement.keys():
                if "userLocalTime" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["userLocalTime"]
                elif "timeUtc" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["timeUtc"]
        case _:
            print(f'Found statement: {statement["answerScheme"]} - {statement["statementId"]} -  {statement}')


    return field
        

def returnEntry(entry):
    keys = entry.keys()
    finalFields = {}
    if "statementId" in keys:
        finalFields = finalFields | extractStatement(entry)
    else:
        for key in keys:
            
            if key.lower() in exceptions:
                continue
            
            typing = type(entry[key]).__name__
            match typing:
                case "str":
                    finalFields[key] = entry[key]
                    
                case "int":
                    finalFields[key] = entry[key]

                case "bool":
                    finalFields[key] = entry[key]

                case "float":
                    finalFields[key] = entry[key]
                    
                case "dict":
                    if "displayText" in entry[key].keys():
                        finalFields[key] = entry[key]["displayText"]
                        continue
                    else:
                        finalFields = finalFields | returnEntry(entry[key])
                    
                case "list":
                    for item in entry[key]:
                        if type(item).__name__ == "dict":
                            finalFields = finalFields | returnEntry(item)
                        

                case _:
                    print(typing)


    
    return finalFields

In [ ]:
totalData = []

if data["entries"] is not None:
    for entry in data["entries"]:
        test = returnEntry(entry)
        totalData.append(test)
        
df = pd.DataFrame(totalData)
df

In [ ]:
for col in df.columns:
    non_null = df[col].dropna()

    if len(non_null) > 0 and non_null.isin([True, False]).all():
        df[col] = df[col].fillna(False).astype(bool)

bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns

df[bool_cols] = df[bool_cols].fillna(False)


df = df.replace(r'^\s*$', np.nan, regex=True)


df.to_csv("clientRaw.csv", sep=";", index=False)
df

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
df = df.dropna(axis=1, thresh=int(0.9 * len(df)))

df = df.dropna()

In [ ]:
df

In [ ]:
min_unique = 2
max_unique = len(df) * 0.9


n_unique = df.nunique()

cols_to_drop = n_unique[
    (n_unique < min_unique) | (n_unique > max_unique)
].index

cols_to_drop = [col for col in cols_to_drop if col != "id"]

df = df.drop(columns=cols_to_drop)

df

In [ ]:
if "levelOfCare" in df.columns:
	df["levelOfCare"] = df["levelOfCare"].replace({
		"Pflegestufe 1": 1,
		"Pflegestufe 2": 2,
		"Pflegestufe 3": 3,
		"Pflegestufe 4": 4,
		"Pflegestufe 5": 5,
		"Pflegestufe 6": 6,
		"Pflegestufe 7": 7,
		"Keine Pflegestufe": 0
	})
if "admin-zone" in df.columns:
	df = pd.get_dummies(df, columns=["admin-zone"], prefix="zone", drop_first=True)
if "genderId" in df.columns:
	df = pd.get_dummies(df, columns=["genderId"], prefix="gender", drop_first=True)
if "admin-packet" in df.columns:
	df = pd.get_dummies(df, columns=["admin-packet"], prefix="paket", drop_first=True)
if "vertragsdatum" in df.columns:
	df["vertragsdatum"] = df["vertragsdatum"].astype(str).str[:8].astype(int)
if "betreuungsbeginn" in df.columns:
	df["betreuungsbeginn"] = df["betreuungsbeginn"].astype(str).str[:8].astype(int)

In [ ]:
cols = df.columns.drop("id")
df[cols] = df[cols].astype(int)

In [ ]:
df

In [ ]:
correlation_matrix = df.drop(columns=["id"]).corr()

In [ ]:
import matplotlib.pyplot as plt

corr = df.drop(columns=["id"]).corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
plt.show()

In [ ]:
def remove_correlated_features(df, threshold=0.8, exclude=["id"]):

    result = df.copy()
    dropped = []

    while True:

        features = result.drop(
            columns=exclude,
            errors="ignore"
        )

        corr = features.corr().abs()

        # Remove diagonal
        np.fill_diagonal(corr.values, 0)

        # Find strongest remaining correlation
        max_corr = corr.max().max()

        if max_corr <= threshold:
            break

        # Find the pair
        var1, var2 = corr.stack().idxmax()

        # Mean absolute correlation with all other variables
        score1 = corr.loc[var1].mean()
        score2 = corr.loc[var2].mean()

        # Drop the more redundant variable
        if score1 > score2:
            drop = var1
        else:
            drop = var2

        dropped.append({
            "dropped": drop,
            "var1": var1,
            "var2": var2,
            "pair_corr": max_corr,
            "score_var1": score1,
            "score_var2": score2
        })

        result = result.drop(columns=drop)

    return result, pd.DataFrame(dropped)

In [ ]:
df_reduced, dropped = remove_correlated_features(
    df,
    threshold=0.7
)

print(dropped)

In [ ]:
import matplotlib.pyplot as plt

corr = df_reduced.drop(columns=["id"]).corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
plt.show()

In [ ]:
import gower


X = df_reduced.drop(
    columns=["id"],
    errors="ignore"
).copy()

# Convert all numeric columns to float
numeric_cols = X.select_dtypes(include=np.number).columns
X[numeric_cols] = X[numeric_cols].astype(float)

bool_cols = X.select_dtypes(include=["bool", "boolean"]).columns
X[bool_cols] = X[bool_cols].astype(float)

distance_matrix = gower.gower_matrix(X)

In [ ]:
def kmedoids_precomputed(
    distance_matrix,
    n_clusters,
    max_iter=100,
    random_state=42
):
    D = np.asarray(distance_matrix)

    n_samples = D.shape[0]

    if D.shape[0] != D.shape[1]:
        raise ValueError("distance_matrix must be square")

    rng = np.random.default_rng(random_state)

    # Random initial medoids
    medoid_indices = rng.choice(
        n_samples,
        size=n_clusters,
        replace=False
    )

    for _ in range(max_iter):

        # Assign every observation to nearest medoid
        labels = np.argmin(
            D[:, medoid_indices],
            axis=1
        )

        new_medoids = medoid_indices.copy()

        for cluster_id in range(n_clusters):

            cluster_indices = np.where(
                labels == cluster_id
            )[0]

            # Empty cluster protection
            if len(cluster_indices) == 0:
                continue

            # Distances among members of this cluster
            cluster_distances = D[
                np.ix_(
                    cluster_indices,
                    cluster_indices
                )
            ]

            # Point minimizing total within-cluster distance
            costs = cluster_distances.sum(axis=1)

            best_local_index = np.argmin(costs)

            new_medoids[cluster_id] = (
                cluster_indices[best_local_index]
            )

        # Stop if medoids no longer change
        if np.array_equal(
            np.sort(new_medoids),
            np.sort(medoid_indices)
        ):
            medoid_indices = new_medoids
            break

        medoid_indices = new_medoids

    # Final labels
    labels = np.argmin(
        D[:, medoid_indices],
        axis=1
    )

    # Equivalent idea to sklearn-extra inertia_
    inertia = np.sum(
        np.min(
            D[:, medoid_indices],
            axis=1
        )
    )

    return {
        "labels": labels,
        "medoid_indices": medoid_indices,
        "inertia": inertia
    }

In [ ]:
models = {}

for k in range(2, 7):

    model = kmedoids_precomputed(
        distance_matrix,
        n_clusters=k,
        random_state=42
    )

    models[k] = {
        "model": model,
        "labels": model["labels"]
    }

In [ ]:
from sklearn.metrics import silhouette_score

results = []

for k in range(2, 7):

    model = kmedoids_precomputed(
        distance_matrix,
        n_clusters=k,
        random_state=42
    )

    labels = model["labels"]

    silhouette = silhouette_score(
        distance_matrix,
        labels,
        metric="precomputed"
    )

    results.append({
        "k": k,
        "silhouette": silhouette,
        "inertia": model["inertia"]
    })

results = pd.DataFrame(results)

print(results)

In [ ]:
best_k = results.loc[
    results["silhouette"].idxmax(),
    "k"
]

print("Best k:", best_k)

In [ ]:
df_clustered = df_reduced.copy()

df_clustered["cluster"] = models[best_k]["labels"]

In [ ]:
df_clustered.value_counts("cluster")

In [ ]:
profile = (
    df_clustered.drop(columns=["id"])
    .groupby("cluster")
    .mean()
    .T
)

print(profile)

In [ ]:
df_clustered.to_csv("clients.csv", sep=";", index=False)